# AgentBench — Database 환경 재현: 모델 3개 비교

**논문**: Liu et al., *AgentBench: Evaluating LLMs as Agents*, ICLR 2024

Colab에서 **런타임 유형: GPU (T4)**로 실행하세요.

## 0. 환경 설정

In [ ]:
# Colab에서 실행 시: 저장소를 클론합니다.
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    REPO_URL = "https://github.com/gksmfly/Agentbench-Mini-Eval.git"
    if not os.path.exists("Agentbench-Mini-Eval"):
        !git clone {REPO_URL}
    %cd Agentbench-Mini-Eval/project_db
    !pip install -q requests pandas matplotlib transformers accelerate bitsandbytes
else:
    # 로컬(맥/리눅스)에서 실행 시: 이미 project_db/notebooks 안에 있다고 가정
    os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None

In [ ]:
import json
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

DATA_FILE = ROOT / "data" / "sample_cases.jsonl"
METRICS_FILE = ROOT / "results" / "metrics.csv"

samples = [json.loads(l) for l in open(DATA_FILE)]
print(f"toy 샘플 {len(samples)}개 로드 완료")

def get_key(name: str, required: bool = True) -> str:
    """우선순위: 이미 설정된 환경변수 -> Colab Secrets(발표 전에 미리 등록) -> getpass 수동 입력"""
    if os.environ.get(name):
        return os.environ[name]
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    if required:
        return getpass(f"{name}: ")
    return getpass(f"{name} (없으면 그냥 Enter): ")

# 발표 전 미리 준비해두는 걸 추천:
#   Colab 왼쪽 사이드바(열쇠 아이콘) -> Secrets -> OPENAI_API_KEY / GROQ_API_KEY 등록 + "노트북 액세스" 허용
# 그러면 발표 중엔 이 셀이 조용히 통과하고, 라이브로 키를 입력할 필요가 없습니다.
os.environ["OPENAI_API_KEY"] = get_key("OPENAI_API_KEY")
groq_key = get_key("GROQ_API_KEY", required=False)
if groq_key:
    os.environ["GROQ_API_KEY"] = groq_key

## 1. 핵심 아이디어 최소 재현 — 모델 3개 라이브 비교

실제 300문제 본실험은 THUDM/AgentBench 프레임워크 + 실제 MySQL로 돌렸고 (섹션 3 참고),
여기서는 그 중 **핵심 한 스텝**(자연어 질문 → SQL 제안)만 떼어내 세 모델에게 동시에 물어봅니다.

- `gpt-3.5-turbo`, `gpt-4o` → OpenAI API 호출
- `llama-3.1-8b-instant` → Groq API 호출 (본실험에 실제 사용한 것과 동일 경로)

In [ ]:
from baseline import build_user_prompt, call_llm, SYSTEM_PROMPT

def demo(index: int, model: str):
    entry = samples[index]
    question = build_user_prompt(entry)
    history = [
        {"role": "user", "content": SYSTEM_PROMPT},
        {"role": "assistant", "content": "Ok."},
        {"role": "user", "content": question},
    ]
    reply = call_llm(model, history)
    print(f"--- {model} ---\n{reply}\n")

entry = samples[6]
print(f"[table] {entry['table']['table_name']}  [type] {entry['type']}")
print(f"[question]\n{build_user_prompt(entry)}\n")

for model in ["gpt-3.5-turbo", "gpt-4o", "llama-3.1-8b-instant"]:
    demo(6, model)

print(f"[gold] {entry.get('label') or entry.get('answer_md5')}")

### 1.1 보너스: 오픈소스 모델을 API 없이 Colab GPU에 직접 로드

위에서는 Llama를 Groq API로 호출했지만, Colab의 무료 T4 GPU에 직접 올려서 돌릴 수도 있습니다.
"오픈소스 = 내 GPU만 있으면 완전 무료"라는 걸 실제로 보여주는 선택 데모입니다 (본실험 채점에는 안 씀).

`meta-llama/Llama-3.1-8B-Instruct`는 게이트 모델이라 huggingface.co 라이선스 동의 + 토큰이 필요합니다.
번거로우면 건너뛰어도 됩니다 (섹션 1의 API 데모로 핵심은 충분히 보여줍니다).

In [ ]:
import torch
print("GPU 사용 가능:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(없음, CPU로 진행 시 매우 느릴 수 있음)")

RUN_LOCAL_GPU_DEMO = False  # True로 바꾸면 아래 셀 실행 (HuggingFace 토큰 + 모델 라이선스 동의 필요)

In [ ]:
if RUN_LOCAL_GPU_DEMO:
    from huggingface_hub import login
    login(getpass("HF_TOKEN: "))

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")

    entry = samples[6]
    question = build_user_prompt(entry)
    chat = [
        {"role": "user", "content": SYSTEM_PROMPT},
        {"role": "assistant", "content": "Ok."},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(chat, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(inputs, max_new_tokens=300, do_sample=False)
    print(tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))
else:
    print("RUN_LOCAL_GPU_DEMO = False — 건너뜀 (섹션 1의 Groq API 데모로 대체됨)")

## 2. 실제 300문제 본실험 결과 — 3개 모델 비교

각 모델을 THUDM/AgentBench 프레임워크(Docker + 실제 MySQL + 멀티라운드 에이전트 루프)로
`dbbench-std` 300문제 전체에 대해 실행한 결과입니다. `src/evaluate.py`가 원본 채점 로직을
재구현해 각 모델의 `results/raw/dbbench_std_overall_<model>.json`과 수치가 일치함을 확인했습니다.

In [ ]:
metrics = pd.read_csv(METRICS_FILE)
metrics[metrics["metric"] == "overall_cat_accuracy (Success Rate)"]

In [ ]:
import matplotlib.pyplot as plt

models = ["gpt-3.5-turbo", "gpt-4o", "llama-3.1-8b"]

# (a) SELECT/INSERT/UPDATE accuracy per model
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

acc = metrics[metrics["metric"].isin(["SELECT_accuracy", "INSERT_accuracy", "UPDATE_accuracy"])]
pivot = acc.pivot(index="metric", columns="model", values="value").reindex(models, axis=1)
pivot.plot.bar(ax=axes[0])
axes[0].set_title("Accuracy by SQL type")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=0)

# (b) completion status breakdown per model
status = metrics[metrics["metric"].str.startswith("status:")].copy()
status["status"] = status["metric"].str.replace("status:", "")
status_pivot = status.pivot(index="model", columns="status", values="value").reindex(models).fillna(0)
status_pivot.plot.bar(stacked=True, ax=axes[1])
axes[1].set_title("Run status breakdown")
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=7, loc="upper right")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

### 2.1 추가 분석: 비용·속도·SQL 실행 에러율·응답 길이

정확도만으로는 안 보이는 실사용 관점 지표입니다. `src/analyze.py`가 계산한 `results/analysis.csv`를 읽습니다.

In [ ]:
ANALYSIS_FILE = ROOT / "results" / "analysis.csv"
analysis = pd.read_csv(ANALYSIS_FILE)

key_metrics = [
    "estimated_cost_usd_300_samples",
    "estimated_cost_usd_per_correct_answer",
    "sql_execution_error_rate",
    "avg_agent_response_chars",
]
summary = analysis[analysis["metric"].isin(key_metrics)].pivot(
    index="metric", columns="model", values="value"
).reindex(models, axis=1)
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
summary.loc[["sql_execution_error_rate"]].T.plot.bar(ax=axes[0], legend=False)
axes[0].set_title("SQL execution error rate (per sample)")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=15)

summary.loc[["estimated_cost_usd_per_correct_answer"]].T.plot.bar(ax=axes[1], legend=False, color="darkorange")
axes[1].set_title("Estimated cost per correct answer (USD)")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

## 3. 발견한 이슈 4가지

### 3.1 MySQL 버전 드리프트 — 채점 자체가 실패하던 버그
`mysql:latest`가 논문 시점(2023, MySQL 8.0)과 지금(2026, MySQL 9.x) 사이 이동하면서 `MD5()` 함수 처리
방식이 바뀌어, INSERT/UPDATE 채점 쿼리가 `FUNCTION <db>.MD5 does not exist` 에러로 실패했습니다.
`mysql:8.0` 고정으로 해결 (INSERT 0%→37%, UPDATE 0%→71%).

### 3.2 gpt-4o의 "컨텍스트 한도 초과" 오분류 (25.7%)
gpt-4o는 완료율 53.3%, 25.7%가 "agent context limit"로 분류됩니다. gpt-4o는 128k 토큰급이라 실제
컨텍스트 초과 가능성은 낮습니다. `check_context_limit()`이 에러 응답 텍스트의 키워드 매칭만으로
판단하는 허술한 로직이라, 레이트리밋 등 다른 에러까지 오분류할 수 있음을 코드로 확인했습니다.

추가 분석이 이 가설을 뒷받침합니다: gpt-4o의 평균 응답 길이(259자)가 가장 길고, 이 오분류가 특히
여러 라운드가 필요한 INSERT 문제에 집중됩니다(100개 중 41개) — 매 라운드 사고과정이 누적되며
발생하는 것으로 보입니다.

### 3.3 llama-3.1-8b의 완전 실패 (완료율 0%) — 포맷 미준수
82.3%가 포맷 미준수로 실패합니다. 지시된 고정 문구 `Action: Operation` 대신 `Action: Update`처럼
SQL 종류에 맞춰 말을 바꿔 써서 정규식 매칭에 실패합니다. 논문의 "약한 모델일수록 지시사항을 못
따른다"는 핵심 주장을 2026년 소형 오픈소스 모델로 재현한 사례입니다.

### 3.4 정확도가 다가 아니다
gpt-3.5-turbo는 SQL 실행 에러율이 50.7%로 가장 높은데도 재시도로 복구해, 최종 Success Rate(44.67%)는
gpt-4o(47.33%)와 비슷합니다. 반면 정답 하나당 비용은 gpt-4o가 gpt-3.5-turbo의 3.4배($0.0113 vs
$0.0033)입니다. 정확도 차이는 3%p뿐인데 비용은 훨씬 큽니다.